In [1]:
import numpy as np
from scipy.linalg import expm
from functions import *
from pprint import pprint

In [2]:
def states(n_sites, n_elec_up, n_elec_dn):
    return np.zeros((n_sites, n_elec_up + n_elec_dn))


def hubbard_1D_ham_kintic(n_sites, t=1, pbc=False):
    K = np.zeros((n_sites, n_sites))

    for i in range(n_sites):
        if pbc:
            j = (i + 1) % n_sites
        else:
            if (i + 1) < n_sites:
                j = i + 1
            else:
                continue
        
        K[i, j] = -t
        K[j, i] = -t

    return K



In [3]:
n_sites = 2
n_elec_up = 1
n_elec_dn = 1
n_particles = n_elec_up + n_elec_dn
t = 1.0
delta_tau = 0.05
U = 4.0
n_walkers = 10
n_blocks = 10 # What are blocks?

In [4]:
Phi_t = states(n_sites, n_elec_up, n_elec_dn)
K = hubbard_1D_ham_kintic(n_sites, pbc=False)
Proj_k_half = expm(-0.5 * delta_tau * K); 

K_evals, K_evecs = np.linalg.eigh(K)
Phi_t = np.hstack((K_evecs[:, 0:n_elec_up], K_evecs[:, 0:n_elec_dn]))

E_K_t = np.sum(K_evals[0 : n_elec_up]) + np.sum(K_evals[0 : n_elec_dn])

n_r_up = np.diag(Phi_t[:, 0:n_elec_up] @ Phi_t[:, 0:n_elec_up].conj().T)
n_r_dn = np.diag(Phi_t[:, n_elec_up:n_particles] @ Phi_t[:, n_elec_up:n_particles].conjugate().transpose())
E_V_t = U * n_r_up.conj().T @ n_r_dn
E_t = E_K_t + E_V_t

In [5]:
Phi = np.zeros((n_walkers, n_sites, n_particles)) # Walker ensamble
for i in range(n_walkers):
    Phi[i, :, :] = Phi_t

weights = np.ones(n_walkers)
overlap = np.ones(n_walkers)

E_block = np.zeros(n_blocks)
W_block = np.zeros(n_blocks)

In [ ]:
fac_norm = (np.real(E_t) - 0.5 * U * n_particles) * delta_tau # ????
gamma = np.arccosh(np.exp(0.5 * delta_tau * U))
auxiliary_field = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        auxiliary_field[i, j] = np.exp(gamma * (-1)**(i+j))

# auxiliary_field

array([[1.57570546, 0.63463637],
       [0.63463637, 1.57570546]])

In [7]:
# print(K_evals)
# print(K_evecs)
# print(Proj_k_half)
print(Phi_t)
print(E_t)
print(Phi_t[:, 0:n_elec_up])
print(Phi_t[:, n_elec_up:n_particles])

[[-0.70710678 -0.70710678]
 [-0.70710678 -0.70710678]]
-8.881784197001252e-16
[[-0.70710678]
 [-0.70710678]]
[[-0.70710678]
 [-0.70710678]]


In [8]:


print(n_r_up)
print(n_r_dn)

[0.5 0.5]
[0.5 0.5]
